<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著 <a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》（从零构建大语言模型）</a> 一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 通过 Ollama 使用 Llama 3 模型在本地评估 instruction 响应

- 本 notebook 通过 ollama 使用 80 亿参数的 Llama 3 模型，基于包含生成模型响应的 JSON 格式 dataset 来评估 instruction fine-tuned LLM 的响应，例如：



```python
{
    "instruction": "What is the atomic number of helium?",
    "input": "",
    "output": "The atomic number of helium is 2.",               # <-- 测试集中给定的目标答案
    "model 1 response": "\nThe atomic number of helium is 2.0.", # <-- 某个 LLM 的响应
    "model 2 response": "\nThe atomic number of helium is 3."    # <-- 第二个 LLM 的响应
},
```

- 此代码不需要 GPU，可在笔记本电脑上运行（已在 M3 MacBook Air 上测试）

In [1]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## 安装 Ollama 并下载 Llama 3

- Ollama 是一个高效运行 LLM 的应用程序
- 它是 [llama.cpp](https://github.com/ggerganov/llama.cpp) 的封装，后者用纯 C/C++ 实现 LLM 以最大化效率
- 请注意，它是用于让 LLM 生成文本（推理）的工具，而非用于训练或 fine-tune LLM
- 在运行以下代码之前，请访问 [https://ollama.com](https://ollama.com) 并按照说明安装 ollama（例如点击"Download"按钮，下载适合你操作系统的 ollama 应用程序）

- 对于 macOS 和 Windows 用户，点击下载的 ollama 应用程序；如果提示安装命令行工具，选择"是"
- Linux 用户可以使用 ollama 网站上提供的安装命令

- 通常，在命令行使用 ollama 之前，我们需要启动 ollama 应用程序或在单独的终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">

**图 7.1**：在终端中运行 `ollama serve` 启动 Ollama 服务。Ollama 需要一个后台服务进程来处理来自 Python REST API 或命令行的模型推理请求。

```
┌──────────────────────────────────────────────────┐
│                  Ollama 架构                       │
│                                                    │
│   ┌──────────┐     ┌─────────────┐                │
│   │ 终端/Python│────▶│ ollama serve │               │
│   │  REST API │     │  (后台服务)   │               │
│   └──────────┘     └──────┬──────┘                │
│                           │                        │
│                    ┌──────▼──────┐                │
│                    │  llama.cpp   │                │
│                    │ (C/C++ 推理) │                │
│                    └──────┬──────┘                │
│                           │                        │
│                    ┌──────▼──────┐                │
│                    │  Llama 3 8B  │                │
│                    │   (模型权重)  │                │
│                    └─────────────┘                │
└──────────────────────────────────────────────────┘
```

> **核心结论**：Ollama 通过 llama.cpp 提供高效的本地 LLM 推理，无需 GPU 即可在笔记本电脑上运行。


- 在 ollama 应用程序或 `ollama serve` 运行的情况下，在另一个终端的命令行中执行以下命令来试用 80 亿参数的 Llama 3 模型（该模型占用 4.7 GB 存储空间，首次执行时会自动下载）

```bash
# 8B model
ollama run llama3
```


输出如下所示：

```
$ ollama run llama3
pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
removing any unused layers 
success 
```

- 注意 `llama3` 指的是 instruction fine-tuned 的 80 亿参数 Llama 3 模型

- 或者，如果你的机器支持，你也可以将 `llama3` 替换为 `llama3:70b` 来使用更大的 700 亿参数 Llama 3 模型

- 下载完成后，你将看到一个命令行提示符，允许你与模型对话

- 尝试输入类似"What do llamas eat?"的提示，应该会返回类似以下的输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- 你可以使用输入 `/bye` 结束此会话

## 使用 Ollama 的 REST API

- 现在，另一种与模型交互的方式是通过以下函数在 Python 中使用其 REST API
- 在运行本 notebook 中的后续单元格之前，请确保 ollama 仍在运行，如上所述：
  - 在终端中运行 `ollama serve`
  - 或运行 ollama 应用程序
- 接下来，运行以下代码单元格来查询模型

- 首先，让我们用一个简单的示例测试 API，确保它按预期工作：

In [2]:
import json
import requests


def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Send the POST request
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily feed on plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: High-quality hay, such as alfalfa or timothy hay, is a staple in a llama's diet. They enjoy the sweet taste and texture of fresh hay.
3. Grains: Llamas may receive grains like oats, barley, or corn as part of their daily ration. However, it's essential to provide these grains in moderation, as they can be high in calories.
4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.
5. Minerals: Llamas require access to mineral supplements, which help maintain their overall health and well-being.

In the wild, llamas might also eat:

1. Leaves: They'll munch on leaves from trees and shrubs, including plants like willow, alder, and birch.
2. Bark: In some cases, ll

## 加载 JSON 条目

- 现在，让我们进入数据评估部分
- 这里，我们假设已将测试 dataset 和模型响应保存为 JSON 文件，可以按如下方式加载：

In [3]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 100


- 此文件的结构如下，其中包含测试 dataset 中给定的响应（`'output'`）以及两个不同模型的响应（`'model 1 response'` 和 `'model 2 response'`）：

In [4]:
json_data[0]

{'instruction': 'Calculate the hypotenuse of a right triangle with legs of 6 cm and 8 cm.',
 'input': '',
 'output': 'The hypotenuse of the triangle is 10 cm.',
 'model 1 response': '\nThe hypotenuse of the triangle is 3 cm.',
 'model 2 response': '\nThe hypotenuse of the triangle is 12 cm.'}

- 下面是一个小的工具函数，用于格式化输入，便于后续可视化：

In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 现在，让我们使用 ollama API 来比较模型响应（我们仅评估前 5 个响应以便直观比较）：

In [6]:
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model 1 response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The hypotenuse of the triangle is 10 cm.

Model response:
>> 
The hypotenuse of the triangle is 3 cm.

Score:
>> I'd score this response as 0 out of 100.

The correct answer is "The hypotenuse of the triangle is 10 cm.", not "3 cm.". The model failed to accurately calculate the length of the hypotenuse, which is a fundamental concept in geometry and trigonometry.

-------------------------

Dataset response:
>> 1. Squirrel
2. Eagle
3. Tiger

Model response:
>> 
1. Squirrel
2. Tiger
3. Eagle
4. Cobra
5. Tiger
6. Cobra

Score:
>> I'd rate this model response as 60 out of 100.

Here's why:

* The model correctly identifies two animals that are active during the day: Squirrel and Eagle.
* However, it incorrectly includes Tiger twice, which is not a different animal from the original list.
* Cobra is also an incorrect answer, as it is typically nocturnal or crepuscular (active at twilight).
* The response does not meet the instruction to provide three different animals

- 注意响应非常冗长；为了量化哪个模型更好，我们只想返回分数：

In [7]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue

    return scores

- 现在让我们将此评估应用到整个 dataset 并计算每个模型的平均分（在 M3 MacBook Air 笔记本电脑上每个模型大约需要 1 分钟）
- 注意 ollama 在不同操作系统上并非完全确定性（截至撰写时），因此你得到的数字可能与下面显示的略有不同

In [8]:
from pathlib import Path

for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model)
    print(f"\n{model}")
    print(f"Number of scores: {len(scores)} of {len(json_data)}")
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")

    # Optionally save the scores
    save_path = Path("scores") / f"llama3-8b-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)

Scoring entries: 100%|████████████████████████| 100/100 [01:02<00:00,  1.59it/s]



model 1 response
Number of scores: 100 of 100
Average score: 78.48



Scoring entries: 100%|████████████████████████| 100/100 [01:10<00:00,  1.42it/s]


model 2 response
Number of scores: 99 of 100
Average score: 64.98



- 基于上述评估，我们可以说第 1 个模型优于第 2 个模型